In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
from scipy.stats import mode



r_seed = 42

from pathlib import Path
WORKDIR = str(Path.cwd().parent) + '/'  # raiz do repositório
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/logs/"+data_hora

#pd.set_option('display.float_format', lambda x: f'{x:.17f}')

In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [3]:
## get dataset
pd.set_option("display.precision", 9)

# select only sensor data from smart watches
adl_data = np.load(WORKDIR+'/Data/UNIMIB-SHAR/adl_data.npy',allow_pickle=True)

In [4]:
adl_data = pd.DataFrame(adl_data)

In [5]:
adl_data.head()

,0,1,2,3,4,5,6,7,8,9,...,443,444,445,446,447,448,449,450,451,452
0,7.609005324,7.322881624,7.281458000,7.753252975,8.043098785,8.215854029,8.388744746,8.578034336,8.658355949,8.670647745,...,-6.372120389,-6.323450000,-6.474439938,-6.449741844,-6.415009860,-6.514917946,-6.590504364,-6.535513814,-6.453252069,-6.182646800
1,8.507709000,8.480922372,8.747757492,8.863954425,8.863968376,8.458412696,7.782317342,7.412025900,7.589060953,7.709208608,...,-6.785399713,-6.583993264,-6.291559573,-6.472509671,-6.626783193,-6.774499937,-6.929721671,-6.846733584,-6.704523541,-6.513263797
2,8.442040396,8.632097686,8.730012182,8.697697839,8.552649541,8.057507907,0.395887949,-0.090756437,-0.335575011,-0.507582163,...,-6.403487337,-6.450790528,-6.612762268,-6.830471904,-6.872513943,-7.249797590,-7.414934568,-7.278581209,-7.127101560,-6.832013909
3,-11.025565773,-11.714652456,-11.765208033,-9.944633699,-7.660169437,-6.882159548,-7.601299241,-9.161294094,-10.339049523,-11.046978405,...,-7.715562701,-7.764263449,-7.755619207,-8.040818876,-8.091954629,-7.817942600,-7.768973746,-7.687118196,-7.795513790,-7.819322735
4,0.238471558,0.321548855,0.383801880,0.495767001,0.664556218,0.776264244,1.019608510,1.010644705,0.850411656,0.473916180,...,-7.614920052,-7.553617527,-7.381001956,-7.324490962,-7.473059700,-7.441543048,-7.546224051,-7.474857599,-7.441729911,-7.412474949


In [6]:
adl_data.shape

(7579, 453)

In [7]:
adl_labels = np.load(WORKDIR+'/Data/UNIMIB-SHAR/adl_labels.npy')

In [8]:
adl_labels.shape

(7579, 3)

In [9]:
adl_labels = pd.DataFrame(adl_labels)

In [10]:
adl_labels.head()

,0,1,2
0,1,1,1
1,1,1,1
2,1,1,1
3,1,1,2
4,1,1,2


In [11]:
adl_labels[0].value_counts().sort_index()

0
1     153
2     216
3    1738
4    1985
5     921
6     746
7    1324
8     296
9     200
Name: count, dtype: int64

In [12]:
adl_data[453] = adl_labels[1]

In [13]:
adl_data[454] = adl_labels[0]

In [14]:
adl_data.tail()

,0,1,2,3,4,5,6,7,8,9,...,445,446,447,448,449,450,451,452,453,454
7574,0.585587080,1.859555032,3.734279311,4.728035367,4.535471956,2.763820063,1.664585242,1.042932032,0.981383691,1.169803874,...,0.827012845,0.828856703,0.943561793,0.957409100,0.928894049,0.856220301,0.808746593,0.800933296,29,9
7575,1.863566038,1.855972178,1.916614400,2.024786788,1.932982164,1.878294100,1.970762612,1.796263512,1.666530051,1.449341833,...,1.137082500,1.152554273,1.187331000,1.149010700,1.159294558,1.129326186,1.128397043,1.072502139,29,9
7576,1.265169000,1.265169000,1.233159194,1.352462527,1.380130000,1.408954680,1.380490236,1.341809700,1.341809700,1.232875730,...,-0.151204709,-0.092296916,-0.077794966,-0.185326259,-0.223396264,-0.195889966,-0.084953641,-0.088815399,30,9
7577,2.822006028,2.765955781,2.831002885,2.757367269,2.721341000,2.785725200,2.805072048,2.873277031,2.749181971,2.700669242,...,-0.189095156,-0.413160848,-0.310051616,-0.422122180,-0.403629724,-0.357143446,-0.345481570,-0.383801880,30,9
7578,3.334466000,3.334466000,3.274426744,3.334466000,3.354262082,3.336589462,3.202864551,3.216909437,3.359629119,3.311937101,...,-0.338416441,-0.371398306,-0.242217854,-0.336125418,-0.421602963,-0.263005653,-0.307161240,-0.374169320,30,9


In [15]:
adl_data[454].value_counts().sort_index()

454
1     153
2     216
3    1738
4    1985
5     921
6     746
7    1324
8     296
9     200
Name: count, dtype: int64

In [16]:
adl_data[453].value_counts().sort_index()

453
1     255
2     364
3     217
4     176
5     185
6     260
7     211
8     248
9     224
10    179
11    249
12    370
13    228
14    239
15    272
16    322
17    320
18    176
19    171
20    363
21    183
22    307
23    218
24    462
25    242
26    199
27    220
28    193
29    307
30    219
Name: count, dtype: int64

In [17]:
import pandas as pd

def transform_accelerometer_data(df):
    """
    Transforms a dataframe of accelerometer data where each row contains a 1x453 vector
    of accelerometer readings (x, y, z), a subject_id, and a label into a long format dataframe.

    Args:
        df (pd.DataFrame): Input dataframe with shape (n, 455).
                           Columns 0-452: Accelerometer data (x, y, z groups),
                           Column 453: Subject ID,
                           Column 454: Label ID.

    Returns:
        pd.DataFrame: Transformed dataframe with columns ['x', 'y', 'z', 'subject_id', 'label'].
    """
    # Initialize an empty list to store transformed data
    transformed_data = []

    # Iterate through each row in the dataframe
    for index, row in df.iterrows():
        # Extract subject_id and label
        subject_id = row[453]
        label = row[454]
        
        # Iterate through the accelerometer data in steps of 3 (x, y, z)
        for i in range(0, 453, 3):
            x = row[i]
            y = row[i + 1]
            z = row[i + 2]
            
            # Append the new row to the transformed_data list
            transformed_data.append([x, y, z, subject_id, label])

    # Convert the transformed data into a new dataframe
    transformed_df = pd.DataFrame(transformed_data, columns=['x', 'y', 'z', 'subject_id', 'label'])

    return transformed_df


In [18]:
import pandas as pd

# Example function to organize data by blocks of 'atividade'
def organize_blocks_by_atividade(data):
    """
    Organizes the dataframe by extracting blocks of the same 'atividade' for each 'pessoa_x'.
    Blocks are concatenated vertically for each 'atividade' value in descending order.

    Args:
        data (pd.DataFrame): Input dataframe with columns including 'pessoa_x', 'atividade', and others.

    Returns:
        pd.DataFrame: New dataframe organized by 'atividade' blocks.
    """
    organized_data = pd.DataFrame(columns=data.columns)  # Create an empty DataFrame with the same columns

    # Loop through each pessoa_x
    for pessoa_id, pessoa_data in data.groupby('label'):
        # Loop through atividade values from 5 to 0
        for atividade_val in range(10, -1, -1):
            # Get blocks of data for the current 'atividade'
            atividade_block = pessoa_data[pessoa_data['label'] == atividade_val]
            # Append to the organized_data
            organized_data = pd.concat([organized_data, atividade_block], axis=0)

    # Reset the index to maintain sequential indexing in the new dataframe
    organized_data.reset_index(drop=True, inplace=True)

    return organized_data

# Assuming 'data' is already loaded as a DataFrame
# organized_data = organize_blocks_by_atividade(data)



In [19]:
import pandas as pd
import numpy as np

def noise_reduction(df, window_size=5):
    """
    Apply noise reduction using a moving average filter to accelerometer and gyroscope data.

    Parameters:
    df (pd.DataFrame): Input dataframe of shape (n, 10) with the first 6 columns containing raw accelerometer and gyroscope data.
    window_size (int): The size of the moving window for the moving average filter. Default is 5.

    Returns:
    pd.DataFrame: DataFrame with noise-reduced accelerometer and gyroscope data.
    """
    # Ensure the dataframe has at least 6 columns
    if df.shape[1] < 3:
        raise ValueError("Input dataframe must have at least 6 columns.")
    
    # Extract the first 6 columns (accelerometer and gyroscope data)
    acc_gyro_data = df.iloc[:, :3]
    
    # Apply a moving average filter to reduce noise
    acc_gyro_smoothed = acc_gyro_data.rolling(window=window_size, min_periods=1, center=True).mean()
    
    # Replace the first 6 columns with the smoothed data
    df.iloc[:, :3] = acc_gyro_smoothed
    
    return df




In [20]:
transformed_adl_data = transform_accelerometer_data(adl_data)

In [21]:
transformed_adl_data.head()

,x,y,z,subject_id,label
0,7.609005324,7.322881624,7.281458000,1.0,1.0
1,7.753252975,8.043098785,8.215854029,1.0,1.0
2,8.388744746,8.578034336,8.658355949,1.0,1.0
3,8.670647745,8.737630000,8.268111508,1.0,1.0
4,8.344579086,8.653750997,8.568926466,1.0,1.0


In [22]:
transformed_adl_data = noise_reduction(transformed_adl_data)

In [23]:
transformed_adl_data.shape

(1144429, 5)

In [24]:
transformed_adl_data = organize_blocks_by_atividade(transformed_adl_data)

In [25]:
transformed_adl_data.subject_id = transformed_adl_data.subject_id.astype(int)

In [26]:
transformed_adl_data.subject_id.value_counts().sort_index()

subject_id
1     38505
2     54964
3     32767
4     26576
5     27935
6     39260
7     31861
8     37448
9     33824
10    27029
11    37599
12    55870
13    34428
14    36089
15    41072
16    48622
17    48320
18    26576
19    25821
20    54813
21    27633
22    46357
23    32918
24    69762
25    36542
26    30049
27    33220
28    29143
29    46357
30    33069
Name: count, dtype: int64

In [27]:
transformed_adl_data.shape

(1144429, 5)

In [28]:
transformed_adl_data = transformed_adl_data.dropna()

In [29]:
transformed_adl_data.shape

(1144429, 5)

In [30]:
print(transformed_adl_data['x'].iloc[0]) 

7.917001015006904


In [31]:
transformed_adl_data.label.value_counts().sort_index()

label
1.0     23103
2.0     32616
3.0    262438
4.0    299735
5.0    139071
6.0    112646
7.0    199924
8.0     44696
9.0     30200
Name: count, dtype: int64

In [32]:
""" import pandas as pd

# Load the labels.txt file into a DataFrame
labels = pd.read_csv(WORKDIR+'labels.txt', sep=' ', header=None, 
                     names=['experiment', 'user', 'activity', 'start', 'end'])

# Create a 'label' column in data_df with default NaN values
data_df['label'] = pd.NA

# Process each row in the labels DataFrame
for _, row in labels.iterrows():
    # Extract the interval and corresponding user and experiment
    start, end = row['start'], row['end']
    user, experiment = row['user'], row['experiment']
    activity = row['activity']
    
    # Find the index interval in data_df
    interval_mask = (data_df.index >= start) & (data_df.index <= end)
    
    # Verify if user and experiment match
    if all(data_df.loc[interval_mask, 'user'] == user) and \
       all(data_df.loc[interval_mask, 'experiment'] == experiment):
        # Assign the activity label to the 'label' column in the specified interval
        data_df.loc[interval_mask, 'label'] = activity
 """

" import pandas as pd\n\n# Load the labels.txt file into a DataFrame\nlabels = pd.read_csv(WORKDIR+'labels.txt', sep=' ', header=None, \n                     names=['experiment', 'user', 'activity', 'start', 'end'])\n\n# Create a 'label' column in data_df with default NaN values\ndata_df['label'] = pd.NA\n\n# Process each row in the labels DataFrame\nfor _, row in labels.iterrows():\n    # Extract the interval and corresponding user and experiment\n    start, end = row['start'], row['end']\n    user, experiment = row['user'], row['experiment']\n    activity = row['activity']\n    \n    # Find the index interval in data_df\n    interval_mask = (data_df.index >= start) & (data_df.index <= end)\n    \n    # Verify if user and experiment match\n    if all(data_df.loc[interval_mask, 'user'] == user) and        all(data_df.loc[interval_mask, 'experiment'] == experiment):\n        # Assign the activity label to the 'label' column in the specified interval\n        data_df.loc[interval_mask, '

In [33]:
""" import pandas as pd
import numpy as np

def label_sensor_data(data_df, labels_path):
   
    # Create a copy of the dataframe to avoid modifying the original
    df = data_df.copy()
    
    # Initialize the 'label' column with a default value (e.g., -1 or 'unknown')
    df['label'] = -1
    
    # Read the labels file
    try:
        labels_df = pd.read_csv(labels_path, header=None, delim_whitespace=True, 
                                names=['experiment_id', 'user_id', 'activity_id', 'start_point', 'end_point'])
    except Exception as e:
        print(f"Error reading labels file: {e}")
        return df
    
    # Process each line in the labels file
    for _, label_row in labels_df.iterrows():
        # Extract label information
        exp_id = label_row['experiment_id']
        user_id = label_row['user_id']
        activity_id = label_row['activity_id']
        start_point = label_row['start_point']
        end_point = label_row['end_point']
        
        # Find the index interval in result_df
        mask = (
            (df['experiment'] == exp_id) & 
            (df['user'] == user_id) & 
            (df.index >= start_point) & 
            (df.index <= end_point)
        )
        
        # Label the rows in the specified interval
        df.loc[mask, 'label'] = activity_id
    
    return df

 """

' import pandas as pd\nimport numpy as np\n\ndef label_sensor_data(data_df, labels_path):\n   \n    # Create a copy of the dataframe to avoid modifying the original\n    df = data_df.copy()\n    \n    # Initialize the \'label\' column with a default value (e.g., -1 or \'unknown\')\n    df[\'label\'] = -1\n    \n    # Read the labels file\n    try:\n        labels_df = pd.read_csv(labels_path, header=None, delim_whitespace=True, \n                                names=[\'experiment_id\', \'user_id\', \'activity_id\', \'start_point\', \'end_point\'])\n    except Exception as e:\n        print(f"Error reading labels file: {e}")\n        return df\n    \n    # Process each line in the labels file\n    for _, label_row in labels_df.iterrows():\n        # Extract label information\n        exp_id = label_row[\'experiment_id\']\n        user_id = label_row[\'user_id\']\n        activity_id = label_row[\'activity_id\']\n        start_point = label_row[\'start_point\']\n        end_point = labe

In [34]:
""" import pandas as pd

# Read the data_df dataframe (assuming it's already loaded)
# data_df = pd.read_csv(WORKDIR + 'Data/processed/your_file.csv')  # Uncomment this line to load your dataframe

# Read labels.txt file (the labels file should be in the same format as described)
labels_df = pd.read_csv(WORKDIR+'labels.txt', header=None, sep="\s+", names=["experiment", "user", "activity", "start_point", "end_point"])

# Initialize the 'label' column in data_df with NaN (or 0 if you prefer a placeholder)
data_df['label'] = None

# Iterate over the rows in the labels dataframe and map the activities to the data_df
for _, row in labels_df.iterrows():
    experiment_id = row['experiment']
    user_id = row['user']
    activity_id = row['activity']
    start_point = row['start_point']
    end_point = row['end_point']
    
    # Filter the rows in data_df that belong to the same experiment and user
    mask = (data_df['experiment'] == experiment_id) & (data_df['user'] == user_id)
    
    # Update the 'label' column for the corresponding rows based on the sample points
    data_df.loc[mask & (data_df.index >= start_point) & (data_df.index <= end_point), 'label'] = activity_id

# The 'label' column is now populated with the activity labels corresponding to each row
 """

' import pandas as pd\n\n# Read the data_df dataframe (assuming it\'s already loaded)\n# data_df = pd.read_csv(\'your_file.csv\')  # Uncomment this line to load your dataframe\n\n# Read labels.txt file (the labels file should be in the same format as described)\nlabels_df = pd.read_csv(WORKDIR+\'labels.txt\', header=None, sep="\\s+", names=["experiment", "user", "activity", "start_point", "end_point"])\n\n# Initialize the \'label\' column in data_df with NaN (or 0 if you prefer a placeholder)\ndata_df[\'label\'] = None\n\n# Iterate over the rows in the labels dataframe and map the activities to the data_df\nfor _, row in labels_df.iterrows():\n    experiment_id = row[\'experiment\']\n    user_id = row[\'user\']\n    activity_id = row[\'activity\']\n    start_point = row[\'start_point\']\n    end_point = row[\'end_point\']\n    \n    # Filter the rows in data_df that belong to the same experiment and user\n    mask = (data_df[\'experiment\'] == experiment_id) & (data_df[\'user\'] == use

In [35]:
#selecting only activities 0 to 3
transformed_adl_data = transformed_adl_data[transformed_adl_data['label'].isin([1,2,3,8,9])]

In [36]:
# Define the mapping dictionary
mapping_dict = {1:0,2:1,3:2,8:3,9:4}

In [37]:
transformed_adl_data["label"] = transformed_adl_data["label"].replace(to_replace=mapping_dict)

In [38]:
transformed_adl_data.label.value_counts().sort_index()

label
0.0     23103
1.0     32616
2.0    262438
3.0     44696
4.0     30200
Name: count, dtype: int64

In [39]:
transformed_adl_data.label = transformed_adl_data.label.astype(int)

In [40]:
transformed_adl_data.tail()

,x,y,z,subject_id,label
1144424,0.193999847,0.211816414,-0.018270793,30,4
1144425,0.030715088,0.036619102,-0.212599537,30,4
1144426,-0.143836370,-0.152447624,-0.392445027,30,4
1144427,-0.294606713,-0.322009169,-0.385245072,30,4
1144428,-0.284007426,-0.327234367,-0.389056863,30,4


In [41]:
transformed_adl_data_validation = transformed_adl_data[transformed_adl_data['subject_id'].isin([1,6,11,16,21,22,26,30])]

In [42]:
transformed_adl_data_validation.head()

,x,y,z,subject_id,label
0,7.917001015,7.981338249,8.051889326,1,0
1,8.105412697,8.170411186,8.105944872,1,0
2,8.153245975,8.267079148,8.198541190,1,0
3,8.383688565,8.590818513,6.769184863,1,0
4,6.871478034,7.020638820,5.155788480,1,0


In [43]:
transformed_adl_data_validation.shape

(88788, 5)

In [44]:
transformed_adl_data_validation.label.value_counts()

label
2    50585
3    13137
1     9966
4     8003
0     7097
Name: count, dtype: int64

In [45]:
transformed_adl_data_validation.subject_id.value_counts().sort_index()

subject_id
1     12533
6     11778
11     7097
16    16610
21     6795
22    18422
26     5889
30     9664
Name: count, dtype: int64

In [46]:
transformed_adl_data_validation = transformed_adl_data_validation.reset_index(drop = True)

In [47]:
transformed_adl_data_train = transformed_adl_data[~transformed_adl_data['subject_id'].isin([1,6,11,16,21,22,26,30])]


In [48]:
transformed_adl_data_train.shape

(304265, 5)

In [49]:
transformed_adl_data_train.head()

,x,y,z,subject_id,label
906,2.331159889,2.235437198,2.064675625,2,0
907,5.357005175,5.250555973,5.142262288,2,0
908,8.461382963,8.454239541,6.679629834,2,0
909,6.843327553,6.901078292,5.242471621,2,0
910,5.495982478,5.534026206,3.759320563,2,0


In [50]:
transformed_adl_data_train.subject_id.value_counts().sort_index()

subject_id
2     26425
3      6493
4      5889
5      7248
7     10570
8     14194
9     10872
10     7097
12    29596
13     8154
14    10570
15    12231
17    21442
18     7097
19     6946
20    29445
23     7399
24    37297
25    11325
27     7248
28     5889
29    20838
Name: count, dtype: int64

In [51]:
transformed_adl_data_train.label.value_counts().sort_index()

label
0     16006
1     22650
2    211853
3     31559
4     22197
Name: count, dtype: int64

In [52]:
transformed_adl_data_train = transformed_adl_data_train.reset_index(drop = True)

In [53]:
transformed_adl_data_train.shape

(304265, 5)

In [54]:
transformed_adl_data_train.subject_id.value_counts().sort_index()

subject_id
2     26425
3      6493
4      5889
5      7248
7     10570
8     14194
9     10872
10     7097
12    29596
13     8154
14    10570
15    12231
17    21442
18     7097
19     6946
20    29445
23     7399
24    37297
25    11325
27     7248
28     5889
29    20838
Name: count, dtype: int64

In [55]:
grouped_data_organized = transformed_adl_data_train.groupby(['subject_id'])['label'].value_counts().unstack()

In [56]:
print(grouped_data_organized)

label            0       1        2       3       4
subject_id                                         
2           1057.0  1812.0  21593.0  1359.0   604.0
3            906.0   151.0   3322.0  1661.0   453.0
4              NaN   604.0   2416.0  1510.0  1359.0
5           1057.0  1812.0   3322.0   906.0   151.0
7              NaN   755.0   7399.0  1208.0  1208.0
8            302.0     NaN  10117.0  1963.0  1812.0
9            151.0   453.0   7550.0  1208.0  1510.0
10             NaN     NaN   4832.0   302.0  1963.0
12           906.0   755.0  27482.0     NaN   453.0
13           453.0   906.0   3624.0  2567.0   604.0
14           151.0   906.0   8154.0   604.0   755.0
15          1359.0   755.0   7097.0  1812.0  1208.0
17           604.0  2265.0  15553.0  1963.0  1057.0
18          2114.0  1359.0      NaN  1963.0  1661.0
19           906.0  1812.0   2114.0   906.0  1208.0
20          1510.0  1963.0  23556.0  1208.0  1208.0
23           906.0  1057.0   2265.0  2265.0   906.0
24          

In [57]:
print(f"percentage of validation is {transformed_adl_data_validation.shape[0]/transformed_adl_data.shape[0]}")

percentage of validation is 0.22589320015366884


In [58]:
print(f"percentage of validation is {transformed_adl_data_train.shape[0]/transformed_adl_data.shape[0]}")

percentage of validation is 0.7741067998463311


# **Models**

In [59]:
def split_and_shuffle_3d_arrays(array_data, labels_data):
    
    combined = list(zip(array_data, labels_data))


    # Step 2: Separate into four parts based on label value
    # Filtering combined list by labels in each window
    grouped_data = {
        i: [item for item in combined if np.unique(item[1][:, 0])[0] == i] for i in range(3)
    }

    # Step 3: Interlace half of label=1 into half of label=0
    first_third_0 = len(grouped_data[0]) // 3
    half_1 = len(grouped_data[1]) // 3
    interlaced_0 = []

    for i in range(half_0):
        interlaced_0.append(grouped_data[0][i])
        if i < half_1:
            interlaced_0.append(grouped_data[1][i])

    # Step 4: Interlace half of label=3 into half of label=2
    half_2 = len(grouped_data[2]) // 2
    half_3 = len(grouped_data[3]) // 2
    interlaced_2 = []

    for i in range(half_2):
        interlaced_2.append(grouped_data[2][i])
        if i < half_3:
            interlaced_2.append(grouped_data[3][i])

    # Step 5: Interlace last half of label=0 into last half of label=3
    last_half_1 = grouped_data[1][half_1:]
    last_half_0 = grouped_data[0][half_0:]
    interlaced_1 = []

    for i in range(len(last_half_1)):
        interlaced_1.append(last_half_1[i])
        if i < len(last_half_0):
            interlaced_1.append(last_half_0[i])

    # Step 6: Concatenate the three resulting arrays vertically
    combined_result = interlaced_0 + interlaced_2 + interlaced_3

    # Step 7: Unzip the resulting array into separate data and labels
    data_result, labels_result = zip(*combined_result)

    # Converting back to numpy arrays for further processing if needed
    data_result = np.array(data_result)
    labels_result = np.array(labels_result)
    
    return data_result,labels_result



In [60]:
import numpy as np

def category_count_categorical(labels_test_categorical):

    # Assuming your data is stored in a variable called `data`
    # Replace this with the actual data if it's available

    # Sum along the rows to get the count for each category
    category_counts_categorical = np.sum(labels_test_categorical, axis=0)

    # Print the counts
    print("Category counts:", category_counts_categorical)


In [61]:
import numpy as np

# Example array with binary class indications, replace this with your actual array
#array_data = np.random.randint(0, 2, (4027, 7)).astype(float)  # Example data; replace with your own

def count_active_classes(array_data):
    # Sum across each row to get the number of classes present in each row
    active_classes_per_row = np.sum(array_data, axis=1)
    
    # Count unique values in active_classes_per_row to get how many rows have each class count
    unique_counts, count_of_rows = np.unique(active_classes_per_row, return_counts=True)
    
    # Map each unique count of active classes to the number of rows with that count
    count_summary = dict(zip(unique_counts.astype(int), count_of_rows))
    
    return count_summary

# Get the count of active classes for each row
#active_classes_count = count_active_classes(array_data)

# Print the results
""" for active_class_count, row_count in active_classes_count.items():
    print(f"Number of rows with {active_class_count} active classes: {row_count}") """


' for active_class_count, row_count in active_classes_count.items():\n    print(f"Number of rows with {active_class_count} active classes: {row_count}") '

In [62]:
""" import numpy as np

# Function to count occurrences of each category
def count_categories(data):
    counts = {
        '[1, 0, 0, 0]': np.sum((data == [1., 0., 0., 0.]).all(axis=1)),
        '[0, 1, 0, 0]': np.sum((data == [0., 1., 0., 0.]).all(axis=1)),
        '[0, 0, 1, 0]': np.sum((data == [0., 0., 1., 0.]).all(axis=1)),
        '[0, 0, 0, 1]': np.sum((data == [0., 0., 0., 1.]).all(axis=1)),
    }
    return counts

# Count categories before removal
print("Category counts before removal:", count_categories(labels_data_categorical))

# Define the desired class count for the first category
desired_class_count = 2151

# Identify indices of the first class (1, 0, 0, 0) in labels_data_categorical
first_class_indices = np.where((labels_data_categorical == [1., 0., 0., 0.]).all(axis=1))[0]

# Randomly select indices to remove to balance the first class
indices_to_remove = np.random.choice(first_class_indices, size=(len(first_class_indices) - desired_class_count), replace=False)

# Remove the selected indices from both labels_data_categorical and segments_data
labels_data_categorical = np.delete(labels_data_categorical, indices_to_remove, axis=0)
segments_data = np.delete(segments_data, indices_to_remove, axis=0)

# Count categories after removal
print("Category counts after removal:", count_categories(labels_data_categorical))

# Print the shape of the new arrays to verify
print("New shape of labels_data_categorical:", labels_data_categorical.shape)
print("New shape of segments_data:", segments_data.shape)
 """

' import numpy as np\n\n# Function to count occurrences of each category\ndef count_categories(data):\n    counts = {\n        \'[1, 0, 0, 0]\': np.sum((data == [1., 0., 0., 0.]).all(axis=1)),\n        \'[0, 1, 0, 0]\': np.sum((data == [0., 1., 0., 0.]).all(axis=1)),\n        \'[0, 0, 1, 0]\': np.sum((data == [0., 0., 1., 0.]).all(axis=1)),\n        \'[0, 0, 0, 1]\': np.sum((data == [0., 0., 0., 1.]).all(axis=1)),\n    }\n    return counts\n\n# Count categories before removal\nprint("Category counts before removal:", count_categories(labels_data_categorical))\n\n# Define the desired class count for the first category\ndesired_class_count = 2151\n\n# Identify indices of the first class (1, 0, 0, 0) in labels_data_categorical\nfirst_class_indices = np.where((labels_data_categorical == [1., 0., 0., 0.]).all(axis=1))[0]\n\n# Randomly select indices to remove to balance the first class\nindices_to_remove = np.random.choice(first_class_indices, size=(len(first_class_indices) - desired_class_

In [63]:
def delete_segments_not_fully_filled(df_data_windowed,df_label_windowed):
    indexes = np.where(df_label_windowed == -1)[0]
    unique_arr = np.unique(indexes, return_index=True)[0]
    new_df_label_windowed = np.delete(df_label_windowed,unique_arr,axis=0)
    new_df_data_windowed = np.delete(df_data_windowed,unique_arr,axis=0)
    return (new_df_data_windowed,new_df_label_windowed)

In [64]:
import numpy as np

def normalize_segments_to_mode(df_data_windowed, df_label_windowed):
    # Iterate through each window
    for i in range(df_label_windowed.shape[0]):
        # Get the current window
        window = df_label_windowed[i]
        
        # Replace -1 values with the first value of the window
        first_value = window[0]
        print(first_value)
        window = np.where(window == -1, first_value, window)
        
        # Update the window in the array
        df_label_windowed[i] = window
    
    return df_data_windowed, df_label_windowed


In [65]:
def apply_segmentation_no_sliding_window_validation(df_data):
# Define window size and step size
    window_size = 151
    step_size = 30

    # Calculate number of windows
    n_windows = int((df_data.shape[0] / window_size))

    # Create segmented data using  window
    segments = np.zeros((n_windows, window_size, df_data.shape[1]-2),dtype=np.float32)
    targets = np.full((n_windows, window_size, 1),-1)
    for i in range(n_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size
        #populate the 3d array only if the 256 segment has the value in label
         # Label consistency check
        label_consistency = (
            df_data[['label']][start_idx:end_idx].nunique()[0] == 1        
        ) 
        if label_consistency == True:
            segments[i] = df_data[['x', 'y', 'z']][start_idx:end_idx]
            targets[i] = df_data[['label']][start_idx:end_idx]
    return (segments,targets)

In [66]:
def apply_segmentation_no_sliding_window(df_data):
    import numpy as np

    # Define window size and step size
    window_size = 151
    step_size = 30

    # Define the threshold for sufficient transition labels
    transition_threshold = 0.3  # At least 10% of the window size

    # Calculate number of windows
    n_windows = int((df_data.shape[0] / window_size))

    # Create segmented data using sliding window
    segments = np.zeros((n_windows, window_size, df_data.shape[1] - 2), dtype=np.float32)
    targets = np.full((n_windows, window_size, 1), -1)

    for i in range(n_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size

        # Get the first row value for label

        # Label consistency check
        label_consistency = (
            df_data[['label']][start_idx:end_idx].nunique()[0] == 1
        )

        if label_consistency:
            # Populate the 3D array for consistent labels
            segments[i] = df_data[
                ['x', 'y', 'z']
            ][start_idx:end_idx].values
            targets[i] = df_data[['label']][start_idx:end_idx].values                 

    return segments, targets


In [67]:
import numpy as np
import pandas as pd

def apply_segmentation_with_sliding_window(df, window_size=151, step_size=30):
    """
    Segment a DataFrame into windows with specific constraints.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame with columns 'x', 'y', 'z', and a label column
    window_size : int, optional
        Size of each window (default 151)
    step_size : int, optional
        Step size between windows (default 15)
    
    Returns:
    --------
    tuple: (variable_data_array, label_data_array)
        variable_data_array: numpy array of shape (num_windows, window_size, 3)
        label_data_array: numpy array of shape (num_windows, window_size, 1)
    """
    # Extract variable columns and label column
    var_columns = ['x', 'y', 'z', 'subject_id','label']
    label_column = df.columns[df.columns.tolist().index(var_columns[-1])]
    print(label_column)
    # Initialize lists to store valid windows
    variable_windows = []
    label_windows = []
    
    # Iterate through the DataFrame with sliding window
    for start in range(0, len(df) - window_size + 1, step_size):
        # Extract current window
        window = df.iloc[start:start+window_size]
        
        # Check if all labels in the window are the same
        if window[label_column].nunique() == 1:
            # Extract variable data and label data for this window
            var_data = window[var_columns].values
            label_data = window[label_column].values.reshape(-1, 1)
            
            # Append to lists of windows
            variable_windows.append(var_data)
            label_windows.append(label_data)
    
    # Convert lists to numpy arrays
    if variable_windows:
        variable_data_array = np.array(variable_windows)
        label_data_array = np.array(label_windows)
        
        return variable_data_array, label_data_array
    else:
        return np.array([]), np.array([])

# Example usage:
# Assuming 'df' is your input DataFrame
# var_data, label_data = segment_dataframe(df)
# print(var_data.shape, label_data.shape)

In [68]:
#print(data.pessoa_x.unique())

In [69]:
#X_test_data = data[data['pessoa_x'].isin([0,1])]

In [70]:
#print(X_test_data.shape)

In [71]:
#X_test_data = X_test_data.reset_index(drop = True)

In [72]:
#X_test_data.head()

In [73]:
#X_test_data.tail()

In [74]:
#X_test_data.pessoa_x.unique()

In [75]:
#print(f"percentage of validation is {X_test_data.shape[0]/data.shape[0]}")

In [76]:
#X_test_data['atividade'].value_counts()

In [77]:
#X_test_data = X_test_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','magnetrômetro_x','magnetrômetro_y','magnetrômetro_z','atividade']]

In [78]:
#X_train_validation_data = data[~data.pessoa_x.isin([0,1])]

In [79]:
#print(X_train_validation_data.shape)

In [80]:
#X_train_validation_data = X_train_validation_data.reset_index(drop = True)

In [81]:
#X_train_validation_data.head()

In [82]:
#X_train_validation_data.tail()

In [83]:
#X_train_validation_data.pessoa_x.unique()

In [84]:
#print(f"percentage of validation is {X_train_validation_data.shape[0]/data.shape[0]}")

In [85]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

def balance_data_with_smote(data, label_column):
    """
    Balances the class distribution in a DataFrame using SMOTE.

    Parameters:
        data (pd.DataFrame): The input DataFrame containing features and labels.
        label_column (str): The name of the column containing class labels.

    Returns:
        pd.DataFrame: A balanced DataFrame with the same structure as the input.
    """
    # Separate features and labels
    X = data.drop(columns=[label_column])
    y = data[label_column]

    # Check class distribution before SMOTE
    print("Before SMOTE:", Counter(y))

    # Apply SMOTE
    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)

    # Check class distribution after SMOTE
    print("After SMOTE:", Counter(y_smote))

    # Combine features and labels back into a DataFrame
    balanced_data = pd.concat([pd.DataFrame(X_smote, columns=X.columns), 
                               pd.DataFrame(y_smote, columns=[label_column])], axis=1)

    return balanced_data

# Example usage:
# balanced_df = balance_data_with_smote(data, label_column='label')


In [86]:
#X_train_validation_data['atividade'].value_counts()

In [87]:
#X_train_validation_data['atividade'].value_counts()

In [88]:
transformed_adl_data_validation.columns

Index(['x', 'y', 'z', 'subject_id', 'label'], dtype='object')

In [89]:
#transformed_adl_data_validation = transformed_adl_data_validation[['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z','label']]

In [90]:
segments_val,targets_val = apply_segmentation_no_sliding_window_validation(transformed_adl_data_validation)
print(targets_val.shape)
print(segments_val.shape)

(588, 151, 1)
(588, 151, 3)


In [91]:
""" import numpy as np

# Example 3D array (replace with your actual array)
n = 10
array = np.random.randint(-5, 5, (n, 256, 1))  # Replace with your actual array

# Check if the last dimension has any value = -1
has_minus_one = np.any(targets_val[:, :, 0] == 5)

if has_minus_one:
    print("The array contains at least one -1 in the last dimension.")
else:
    print("The array does not contain any -1 in the last dimension.")
 """

' import numpy as np\n\n# Example 3D array (replace with your actual array)\nn = 10\narray = np.random.randint(-5, 5, (n, 256, 1))  # Replace with your actual array\n\n# Check if the last dimension has any value = -1\nhas_minus_one = np.any(targets_val[:, :, 0] == 5)\n\nif has_minus_one:\n    print("The array contains at least one -1 in the last dimension.")\nelse:\n    print("The array does not contain any -1 in the last dimension.")\n '

In [92]:
import numpy as np
from collections import Counter

# Example initialization of the array (replace this with your actual data)
# targets_train_synthetic_transitions = np.random.randint(0, 5, (n, 200, 1))

def count_int_classes_of_array(targets_train):

    # Flatten the last dimension to make it easier to work with
    flattened_targets = targets_train.reshape(targets_train.shape[0], 151)
    print(flattened_targets)

    # Dictionary to store counts of labels where all 200 values are the same
    label_counts = Counter()

    # Iterate through each window
    for window in flattened_targets:
        # Check if all values in the window are the same
        if np.all(window == window[0]):
            # Increment the count for this label
            label_counts[window[0]] += 1

    # Print the final counts
    print("Counts of unique labels where all values in a window are the same:")
    for label, count in label_counts.items():
        print(f"Label {label}: {count}")


In [93]:
""" import numpy as np
from collections import Counter

# Assuming `array` is your 3D array with shape (n, 256, 1)
# Create a sample array for demonstration
n = 10  # Number of windows
array = np.random.randint(0, 10, (n, 256, 1))  # Replace this with your actual array

# Flatten the 3D array into a 1D array
flattened_array = targets_val.flatten()

# Count the unique values
unique_values, counts = np.unique(flattened_array, return_counts=True)

# Combine the results into a dictionary
value_counts = dict(zip(unique_values, counts))

# Display the counts
print("Unique values and their counts:")
for value, count in value_counts.items():
    print(f"Value {value}: Count {count}")
 """

' import numpy as np\nfrom collections import Counter\n\n# Assuming `array` is your 3D array with shape (n, 256, 1)\n# Create a sample array for demonstration\nn = 10  # Number of windows\narray = np.random.randint(0, 10, (n, 256, 1))  # Replace this with your actual array\n\n# Flatten the 3D array into a 1D array\nflattened_array = targets_val.flatten()\n\n# Count the unique values\nunique_values, counts = np.unique(flattened_array, return_counts=True)\n\n# Combine the results into a dictionary\nvalue_counts = dict(zip(unique_values, counts))\n\n# Display the counts\nprint("Unique values and their counts:")\nfor value, count in value_counts.items():\n    print(f"Value {value}: Count {count}")\n '

In [94]:
""" import numpy as np
import pandas as pd

# Assume df is the 3D numpy array of shape (n, 100, 1)
# Example: df = np.random.randint(0, 10, size=(10, 256, 1))

def count_unique_values_per_window(df):
    # Ensure the input is 3D
    if df.ndim != 3 or df.shape[2] != 1:
        raise ValueError("Input must be a 3D array of shape (n, 256, 1)")
    
    # Reshape the third dimension for easier processing
    reshaped_df = df.reshape(df.shape[0], df.shape[1])

    # List to store counts for each window
    unique_counts = []

    for window in reshaped_df:
        # Count unique values and their frequencies
        unique, counts = np.unique(window, return_counts=True)
        unique_counts.append(dict(zip(unique, counts)))

    return unique_counts

# Example usage
#df = np.random.randint(0, 10, size=(5, 256, 1))  # 5 windows of 100 rows with random integers 0-9
result = count_unique_values_per_window(targets_val)

# Convert the result to a DataFrame for better visualization
result_df = pd.DataFrame(result).fillna(0).astype(int)

# Display the result
print(result_df)
 """

' import numpy as np\nimport pandas as pd\n\n# Assume df is the 3D numpy array of shape (n, 100, 1)\n# Example: df = np.random.randint(0, 10, size=(10, 256, 1))\n\ndef count_unique_values_per_window(df):\n    # Ensure the input is 3D\n    if df.ndim != 3 or df.shape[2] != 1:\n        raise ValueError("Input must be a 3D array of shape (n, 256, 1)")\n    \n    # Reshape the third dimension for easier processing\n    reshaped_df = df.reshape(df.shape[0], df.shape[1])\n\n    # List to store counts for each window\n    unique_counts = []\n\n    for window in reshaped_df:\n        # Count unique values and their frequencies\n        unique, counts = np.unique(window, return_counts=True)\n        unique_counts.append(dict(zip(unique, counts)))\n\n    return unique_counts\n\n# Example usage\n#df = np.random.randint(0, 10, size=(5, 256, 1))  # 5 windows of 100 rows with random integers 0-9\nresult = count_unique_values_per_window(targets_val)\n\n# Convert the result to a DataFrame for better v

In [95]:
segments_val,targets_val = delete_segments_not_fully_filled(segments_val,targets_val)
print(targets_val.shape)
print(segments_val.shape)

(588, 151, 1)
(588, 151, 3)


In [96]:
count_active_classes(targets_val)

{0: 47, 151: 66, 302: 335, 453: 87, 604: 53}

In [97]:
count_int_classes_of_array(targets_val)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]]
Counts of unique labels where all values in a window are the same:
Label 0: 47
Label 1: 66
Label 2: 335
Label 3: 87
Label 4: 53


In [98]:
#segments_val,targets_val = normalize_segments_to_mode(segments_val,targets_val)

In [99]:
transformed_adl_data_train.columns

Index(['x', 'y', 'z', 'subject_id', 'label'], dtype='object')

In [100]:
#transformed_adl_data_train = transformed_adl_data_train[['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z','label']]

In [101]:
segments_train,targets_train = apply_segmentation_with_sliding_window(transformed_adl_data_train)
print(targets_train.shape)
print(segments_train.shape)

label
(10118, 151, 1)
(10118, 151, 5)


In [102]:
segments_train = segments_train[:, :, :-2]



In [103]:
print(segments_train)

[[[ 2.33115989  2.2354372   2.06467563]
  [ 5.35700518  5.25055597  5.14226229]
  [ 8.46138296  8.45423954  6.67962983]
  ...
  [-4.96581567 -4.92092358 -4.93678737]
  [-2.26923258 -2.27560608 -2.25246645]
  [ 0.48059324  0.53449799  0.55870052]]

 [[ 7.39843353  7.49784306  7.67298404]
  [ 7.81462456  8.11781814  8.34550973]
  [ 8.46951512  8.68404758  8.72756361]
  ...
  [ 7.04267745  6.83418883  5.8217287 ]
  [ 6.97548073  7.7007702   7.7906155 ]
  [ 7.4995717   7.31605596  7.44489233]]

 [[ 7.06553503  7.04285218  7.0480565 ]
  [ 7.05169922  7.07003161  7.08917627]
  [ 7.09802702  7.09500236  7.06621404]
  ...
  [ 7.65153819  7.6585676   7.61612876]
  [ 7.58321422  7.55451202  7.53629437]
  [ 7.52370647  7.55716199  7.54233949]]

 ...

 [[-2.0898752  -1.84721919 -2.12713068]
  [-1.97073356 -1.84250345 -2.10130884]
  [-1.97136968 -1.80610959 -1.95881428]
  ...
  [-2.22874318 -2.40849171 -2.54727258]
  [-2.59743617 -2.65520322 -2.73122512]
  [-2.77022772 -2.82765204 -2.67830333]]

 [

In [104]:
print(segments_train.shape)

(10118, 151, 3)


In [105]:
count_active_classes(targets_train)

{0: 529, 151: 750, 302: 7057, 453: 1047, 604: 735}

In [106]:
count_int_classes_of_array(targets_train)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]]
Counts of unique labels where all values in a window are the same:
Label 0: 529
Label 1: 750
Label 2: 7057
Label 3: 1047
Label 4: 735


In [107]:
segments_train,targets_train = delete_segments_not_fully_filled(segments_train,targets_train)
print(targets_train.shape)
print(segments_train.shape)

(10118, 151, 1)
(10118, 151, 3)


In [108]:
transformed_adl_data_train.shape

(304265, 5)

In [109]:
segments_train_no_sliding_window,targets_train_no_sliding_window = apply_segmentation_no_sliding_window(transformed_adl_data_train)
print(targets_train_no_sliding_window.shape)
print(segments_train_no_sliding_window.shape)

(2015, 151, 1)
(2015, 151, 3)


In [110]:
count_active_classes(targets_train_no_sliding_window)

{0: 106, 151: 150, 302: 1403, 453: 209, 604: 147}

In [111]:
count_int_classes_of_array(targets_train_no_sliding_window)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]]
Counts of unique labels where all values in a window are the same:
Label 0: 106
Label 1: 150
Label 2: 1403
Label 3: 209
Label 4: 147


In [112]:
segments_train_no_sliding_window,targets_train_no_sliding_window = delete_segments_not_fully_filled(segments_train_no_sliding_window,targets_train_no_sliding_window)
print(targets_train_no_sliding_window.shape)
print(segments_train_no_sliding_window.shape)

(2015, 151, 1)
(2015, 151, 3)


In [113]:
print(segments_train_no_sliding_window)

[[[ 2.3311598   2.2354372   2.0646756 ]
  [ 5.357005    5.250556    5.1422625 ]
  [ 8.461383    8.45424     6.67963   ]
  ...
  [-4.9658155  -4.9209237  -4.936787  ]
  [-2.2692325  -2.2756062  -2.2524664 ]
  [ 0.48059323  0.534498    0.5587005 ]]

 [[ 3.2050369   3.1797287   3.2564554 ]
  [ 6.0015187   6.0215073   4.524683  ]
  [ 7.2220664   7.212758    5.659211  ]
  ...
  [-5.5106163  -5.4624085  -5.438549  ]
  [-2.7172956  -2.6537569  -2.6594934 ]
  [ 0.06504873  0.13041559  0.13445616]]

 [[ 2.923851    1.5442702   1.5157917 ]
  [ 4.2639303   2.84468     2.7904165 ]
  [ 5.504834    4.026771    3.9408998 ]
  ...
  [-4.927425   -4.8165812  -4.871346  ]
  [-5.5744324  -5.540954   -5.6898007 ]
  [-6.4724345  -6.4838676  -4.8003926 ]]

 ...

 [[-1.704787   -1.5139579  -1.3058481 ]
  [ 0.13217968  0.13916318  0.25886992]
  [ 1.6774511   1.7065212   1.7857617 ]
  ...
  [-2.6196537  -2.5871415  -2.5712469 ]
  [-0.87924075 -0.6184055  -0.28429094]
  [ 1.556377    1.8261367   1.8425964 ]]

 [

In [114]:
""" import numpy as np
from collections import Counter

# Assuming `array` is your 3D array with shape (n, 256, 1)
# Create a sample array for demonstration
n = 10  # Number of windows
array = np.random.randint(0, 10, (n, 250, 1))  # Replace this with your actual array

# Flatten the 3D array into a 1D array
flattened_array = targets_val.flatten()

# Count the unique values
unique_values, counts = np.unique(flattened_array, return_counts=True)

# Combine the results into a dictionary
value_counts = dict(zip(unique_values, counts))

# Display the counts
print("Unique values and their counts:")
for value, count in value_counts.items():
    print(f"Value {value}: Count {count}")
 """

' import numpy as np\nfrom collections import Counter\n\n# Assuming `array` is your 3D array with shape (n, 256, 1)\n# Create a sample array for demonstration\nn = 10  # Number of windows\narray = np.random.randint(0, 10, (n, 250, 1))  # Replace this with your actual array\n\n# Flatten the 3D array into a 1D array\nflattened_array = targets_val.flatten()\n\n# Count the unique values\nunique_values, counts = np.unique(flattened_array, return_counts=True)\n\n# Combine the results into a dictionary\nvalue_counts = dict(zip(unique_values, counts))\n\n# Display the counts\nprint("Unique values and their counts:")\nfor value, count in value_counts.items():\n    print(f"Value {value}: Count {count}")\n '

In [115]:
""" import numpy as np
import pandas as pd

# Assume df is the 3D numpy array of shape (n, 256, 1)
# Example: df = np.random.randint(0, 10, size=(10, 256, 1))

def count_unique_values_per_window(df):
    # Ensure the input is 3D
    if df.ndim != 3 or df.shape[2] != 1:
        raise ValueError("Input must be a 3D array of shape (n, 256, 1)")
    
    # Reshape the third dimension for easier processing
    reshaped_df = df.reshape(df.shape[0], df.shape[1])

    # List to store counts for each window
    unique_counts = []

    for window in reshaped_df:
        # Count unique values and their frequencies
        unique, counts = np.unique(window, return_counts=True)
        unique_counts.append(dict(zip(unique, counts)))

    return unique_counts

# Example usage
#df = np.random.randint(0, 10, size=(5, 256, 1))  # 5 windows of 256 rows with random integers 0-9
result = count_unique_values_per_window(targets_val)

# Convert the result to a DataFrame for better visualization
result_df = pd.DataFrame(result).fillna(0).astype(int)

# Display the result
print(result_df)
 """

' import numpy as np\nimport pandas as pd\n\n# Assume df is the 3D numpy array of shape (n, 256, 1)\n# Example: df = np.random.randint(0, 10, size=(10, 256, 1))\n\ndef count_unique_values_per_window(df):\n    # Ensure the input is 3D\n    if df.ndim != 3 or df.shape[2] != 1:\n        raise ValueError("Input must be a 3D array of shape (n, 256, 1)")\n    \n    # Reshape the third dimension for easier processing\n    reshaped_df = df.reshape(df.shape[0], df.shape[1])\n\n    # List to store counts for each window\n    unique_counts = []\n\n    for window in reshaped_df:\n        # Count unique values and their frequencies\n        unique, counts = np.unique(window, return_counts=True)\n        unique_counts.append(dict(zip(unique, counts)))\n\n    return unique_counts\n\n# Example usage\n#df = np.random.randint(0, 10, size=(5, 256, 1))  # 5 windows of 256 rows with random integers 0-9\nresult = count_unique_values_per_window(targets_val)\n\n# Convert the result to a DataFrame for better v

In [116]:
""" import numpy as np

# Example 3D array (replace with your actual array)
n = 10
array = np.random.randint(-5, 5, (n, 256, 1))  # Replace with your actual array

# Check if the last dimension has any value = -1
has_minus_one = np.any(targets_val[:, :, 0] == 5)

if has_minus_one:
    print("The array contains at least one -1 in the last dimension.")
else:
    print("The array does not contain any -1 in the last dimension.")
 """

' import numpy as np\n\n# Example 3D array (replace with your actual array)\nn = 10\narray = np.random.randint(-5, 5, (n, 256, 1))  # Replace with your actual array\n\n# Check if the last dimension has any value = -1\nhas_minus_one = np.any(targets_val[:, :, 0] == 5)\n\nif has_minus_one:\n    print("The array contains at least one -1 in the last dimension.")\nelse:\n    print("The array does not contain any -1 in the last dimension.")\n '

In [117]:
targets_val = targets_val.reshape(targets_val.shape[0],151)

In [118]:
print(targets_val.shape)

(588, 151)


In [119]:
count_active_classes(targets_val)

{0: 47, 151: 66, 302: 335, 453: 87, 604: 53}

In [120]:
print(targets_val)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]]


In [121]:
labels_val = targets_val[:,0]

In [122]:
print(labels_val)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 

In [123]:
print(labels_val.shape)

(588,)


In [124]:
labels_val_categorical = to_categorical(labels_val,num_classes=5)

In [125]:
print(labels_val_categorical)

[[1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]]


In [126]:
print(labels_val_categorical.shape)

(588, 5)


In [127]:
category_count_categorical(labels_val_categorical)

Category counts: [ 47.  66. 335.  87.  53.]


In [128]:
import numpy as np

def augment_data_with_interpolation(variable_data, label_data): 
    def interpolate(window, num_new_windows=1):
        """
        Applies linear interpolation to augment the input window.
        :param window: The input data window (numpy array with shape [151, 3]).
        :param num_new_windows: Number of synthetic windows to create via interpolation.
        :return: List of augmented windows with the same shape as the original.
        """
        augmented_windows = []
        window_length = window.shape[0]  # This should be 151 (time steps)
        feature_count = window.shape[1]  # This should be 3 (features)

        for _ in range(num_new_windows):
            new_window = np.zeros_like(window)  # Start with a copy of the original window

            # Interpolate between each consecutive pair of points for each feature
            for i in range(window_length - 1):
                # For each feature (axis 1), apply linear interpolation
                for f in range(feature_count):
                    interpolation_factor = np.random.uniform(0.1, 0.9)  # Random interpolation factor
                    new_window[i + 1, f] = window[i, f] + interpolation_factor * (window[i + 1, f] - window[i, f])

            augmented_windows.append(new_window)

        return augmented_windows

    # Step 1: Zip the two arrays
    zipped_data = list(zip(variable_data, label_data))

    # To hold augmented data
    augmented_variable_data = []
    augmented_label_data = []
    labels_count = {"label_0": 0, "label_1": 0, "label_2": 0, "label_3": 0, "label_4": 0, "label_5": 0, "label_6": 0}

    # Step 2, 3, 4, 5: Iterate through each window and perform operations
    for var_window, label_window in zipped_data:
        # Check if label in window is unique
        unique_labels = np.unique(label_window)
        if len(unique_labels) == 1 and unique_labels[0] != 2:
            label = unique_labels[0]  # Unique label

            # Define number of synthetic windows based on label
            if label == 1 and labels_count["label_1"] > 6307:
                continue
            elif label == 0 and labels_count["label_0"] > 6528:
                continue
            elif label == 3 and labels_count["label_3"] > 6010:
                continue
            elif label == 4 and labels_count["label_4"] > 6322:
                continue

            if label == 1:
                labels_count["label_1"] += 43
                synthetic_count = 43
            elif label == 0:
                labels_count["label_0"] += 62
                synthetic_count = 62
            elif label == 3:
                labels_count["label_3"] += 29
                synthetic_count = 29
            elif label == 4:
                labels_count["label_4"] += 43
                synthetic_count = 43
            else:
                synthetic_count = 0

            # Generate synthetic windows using INTERPOLATION
            augmented_windows = interpolate(var_window, num_new_windows=synthetic_count)

            for synthetic_window in augmented_windows:
                augmented_variable_data.append(synthetic_window)
                augmented_label_data.append(label_window)  # Use the same label window

    # Step 6: Unzip the augmented arrays
    augmented_variable_data = np.array(augmented_variable_data)
    augmented_label_data = np.array(augmented_label_data)

    return augmented_variable_data, augmented_label_data


In [129]:
transformed_adl_data_train.label.value_counts().sort_index()

label
0     16006
1     22650
2    211853
3     31559
4     22197
Name: count, dtype: int64

In [130]:
import numpy as np

def ocurrence_unique_values(targets_test):

    # Example of 3D array with the format (256, 256, 1)
    #data = np.random.randint(0, 3, (256, 250, 1))

    # Reshape to (256, 256) to simplify the processing
    data_reshaped = targets_test.reshape(targets_test.shape[0], 151)

    # Get unique values per window (each window has 256 rows)
    # Since the second dimension always contains the same value, we can just select the first column for each window
    unique_values = data_reshaped[:, 0]

    # Count occurrences of each unique value (0, 1, 2)
    unique, counts = np.unique(unique_values, return_counts=True)

    # Create a dictionary to show the counts of 0, 1, and 2
    occurrences = dict(zip(unique, counts))
    return occurrences


In [131]:
segments_train_no_sliding_window.shape

(2015, 151, 3)

In [132]:
print(segments_train_no_sliding_window)

[[[ 2.3311598   2.2354372   2.0646756 ]
  [ 5.357005    5.250556    5.1422625 ]
  [ 8.461383    8.45424     6.67963   ]
  ...
  [-4.9658155  -4.9209237  -4.936787  ]
  [-2.2692325  -2.2756062  -2.2524664 ]
  [ 0.48059323  0.534498    0.5587005 ]]

 [[ 3.2050369   3.1797287   3.2564554 ]
  [ 6.0015187   6.0215073   4.524683  ]
  [ 7.2220664   7.212758    5.659211  ]
  ...
  [-5.5106163  -5.4624085  -5.438549  ]
  [-2.7172956  -2.6537569  -2.6594934 ]
  [ 0.06504873  0.13041559  0.13445616]]

 [[ 2.923851    1.5442702   1.5157917 ]
  [ 4.2639303   2.84468     2.7904165 ]
  [ 5.504834    4.026771    3.9408998 ]
  ...
  [-4.927425   -4.8165812  -4.871346  ]
  [-5.5744324  -5.540954   -5.6898007 ]
  [-6.4724345  -6.4838676  -4.8003926 ]]

 ...

 [[-1.704787   -1.5139579  -1.3058481 ]
  [ 0.13217968  0.13916318  0.25886992]
  [ 1.6774511   1.7065212   1.7857617 ]
  ...
  [-2.6196537  -2.5871415  -2.5712469 ]
  [-0.87924075 -0.6184055  -0.28429094]
  [ 1.556377    1.8261367   1.8425964 ]]

 [

In [133]:
segments_train_interpolation,targets_train_interpolation = augment_data_with_interpolation(segments_train_no_sliding_window,targets_train_no_sliding_window)

In [134]:
count_int_classes_of_array(targets_train_interpolation)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]]
Counts of unique labels where all values in a window are the same:
Label 0: 6572
Label 1: 6321
Label 3: 6032
Label 4: 6321


In [135]:
occurrence_crop = ocurrence_unique_values(targets_train_interpolation)

In [136]:
print(occurrence_crop)

{0: 6572, 1: 6321, 3: 6032, 4: 6321}


In [137]:
print(segments_train_interpolation.shape)

(25246, 151, 3)


In [138]:
print(segments_train_interpolation.shape)

(25246, 151, 3)


In [139]:
print(segments_train_interpolation)

[[[ 0.          0.          0.        ]
  [ 3.5403848   4.8301625   4.174654  ]
  [ 7.1542125   5.970792    5.487856  ]
  ...
  [-4.9689484  -4.93669    -4.977607  ]
  [-2.942732   -2.6517732  -2.5509605 ]
  [-0.33692297 -1.1487329  -1.7835622 ]]

 [[ 0.          0.          0.        ]
  [ 4.5149655   3.8838732   3.4168983 ]
  [ 7.918376    5.8559175   5.901876  ]
  ...
  [-4.971913   -4.9271145  -4.9761705 ]
  [-3.2549195  -4.2225933  -4.038833  ]
  [-0.02243611 -1.9653664  -1.7790468 ]]

 [[ 0.          0.          0.        ]
  [ 3.1369696   2.6009471   2.819141  ]
  [ 7.115428    6.6510134   6.393891  ]
  ...
  [-4.9808125  -4.9263153  -4.9583144 ]
  [-3.0939415  -3.75594    -3.9254746 ]
  [-1.1780164   0.22572276 -1.8811468 ]]

 ...

 [[ 0.          0.          0.        ]
  [ 1.5510355   1.5321239   1.4929882 ]
  [ 1.7402005   1.5729984   1.5436116 ]
  ...
  [ 1.1467265   1.1641483   1.1661286 ]
  [ 1.1725345   1.176833    1.1692299 ]
  [ 1.2114589   1.195149    1.1892318 ]]

 [

In [140]:
segments_train_4 = np.vstack((segments_train_interpolation,segments_train))

In [141]:
targets_train_4 = np.vstack((targets_train_interpolation,targets_train))

In [142]:
#segments_train_5 = np.vstack((segments_train_4,segments_train_transitions))

In [143]:
#targets_train_5 = np.vstack((targets_train_4,targets_train_transitions))

In [144]:
segments_train = segments_train_4
targets_train = targets_train_4

In [145]:
# Example arrays (replace these with your actual data)
# segments_train = np.random.random((n, 200, 9))
# targets_train = np.random.random((n, 200, 1))

# Find indices where the label in the last dimension of targets_train is -1.0
indices_to_keep = ~np.any(targets_train.squeeze(-1) == -1.0, axis=1)

# Filter segments_train and targets_train to keep only the rows where labels are not -1.0
segments_train_filtered = segments_train[indices_to_keep]
targets_train_filtered = targets_train[indices_to_keep]

# Print the shapes of the filtered arrays to verify
print(f"Original segments_train shape: {segments_train.shape}")
print(f"Filtered segments_train shape: {segments_train_filtered.shape}")
print(f"Original targets_train shape: {targets_train.shape}")
print(f"Filtered targets_train shape: {targets_train_filtered.shape}")

Original segments_train shape: (35364, 151, 3)
Filtered segments_train shape: (35364, 151, 3)
Original targets_train shape: (35364, 151, 1)
Filtered targets_train shape: (35364, 151, 1)


In [146]:
count_int_classes_of_array(targets_train)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]]
Counts of unique labels where all values in a window are the same:
Label 0: 7101
Label 1: 7071
Label 3: 7079
Label 4: 7056
Label 2: 7057


In [147]:
# Example arrays (replace these with your actual data)
# segments_train = np.random.random((n, 200, 9))
# targets_train = np.random.random((n, 200, 1))

# Find indices where the label in the last dimension of targets_train is -1.0
indices_to_keep = ~np.any(targets_train.squeeze(-1) == -1.0, axis=1)

# Filter segments_train and targets_train to keep only the rows where labels are not -1.0
segments_train_filtered = segments_train[indices_to_keep]
targets_train_filtered = targets_train[indices_to_keep]

# Print the shapes of the filtered arrays to verify
print(f"Original segments_train shape: {segments_train.shape}")
print(f"Filtered segments_train shape: {segments_train_filtered.shape}")
print(f"Original targets_train shape: {targets_train.shape}")
print(f"Filtered targets_train shape: {targets_train_filtered.shape}")

Original segments_train shape: (35364, 151, 3)
Filtered segments_train shape: (35364, 151, 3)
Original targets_train shape: (35364, 151, 1)
Filtered targets_train shape: (35364, 151, 1)


In [148]:
print(segments_train.shape)
print(targets_train.shape)

(35364, 151, 3)
(35364, 151, 1)


In [149]:
count_int_classes_of_array(targets_train_filtered)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]
 [4 4 4 ... 4 4 4]]
Counts of unique labels where all values in a window are the same:
Label 0: 7101
Label 1: 7071
Label 3: 7079
Label 4: 7056
Label 2: 7057


In [150]:
segments_train = segments_train_filtered

In [151]:
targets_train = targets_train_filtered

In [152]:
targets_train = targets_train.reshape(targets_train.shape[0],151)

In [153]:
print(targets_train.shape)

(35364, 151)


In [154]:
labels_train = targets_train[:,0]

In [155]:
labels_train_categorical = to_categorical(labels_train,num_classes=5)

In [156]:
print(labels_train_categorical)

[[1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]]


In [157]:
category_count_categorical(labels_train_categorical)

Category counts: [7101. 7071. 7057. 7079. 7056.]


In [158]:
X_train = segments_train
y_train = labels_train_categorical
X_validation = segments_val
y_validation = labels_val_categorical

In [159]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

(35364, 151, 3)
(35364, 5)
(588, 151, 3)
(588, 5)


In [160]:
category_count_categorical(y_train)

Category counts: [7101. 7071. 7057. 7079. 7056.]


In [161]:
def shuffle_data(variable_data,label_data):
    perm_indices = np.random.permutation(variable_data.shape[0])
    # Shuffle both arrays using the same permutation indices
    shuffled_variable_data = variable_data[perm_indices]
    shuffled_label_data = label_data[perm_indices]
    return (shuffled_variable_data,shuffled_label_data)

In [162]:
#X_train,y_train = shuffle_data(X_train,y_train)
#X_validation,y_validation = shuffle_data(X_validation,y_validation)

In [163]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

(35364, 151, 3)
(35364, 5)
(588, 151, 3)
(588, 5)


In [164]:
X_train = np.array(X_train, dtype=np.float64)
X_validation = np.array(X_validation, dtype=np.float64)

In [165]:
#from tensorflow.keras import regularizers

In [166]:
""" from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Attention, Conv1D, MaxPooling1D, BatchNormalization
from keras.regularizers import l2
import tensorflow as tf

def lstm_attention_sensor():
    inputs = Input(shape=(151, 3))  # Input shape adjusted for accelerometer and gyroscope data
    
    # CNN Layer: Extract local features
    conv_out = Conv1D(filters=96,  # Best parameter for conv_filters
                      kernel_size=5,  # Best parameter for kernel_size
                      activation='relu')(inputs)
    conv_out = MaxPooling1D(pool_size=2)(conv_out)
    conv_out = BatchNormalization()(conv_out)  # Normalize after convolution
    
    # First LSTM layer
    lstm_out = LSTM(96,  # Best parameter for input_units_lstm1
                    kernel_regularizer=l2(0.0014023399602325875),  # Best parameter for l2_reg1
                    recurrent_regularizer=l2(0.0014023399602325875),  # Same as kernel regularizer
                    return_sequences=True)(conv_out)
    
    # Second LSTM layer
    lstm_out_2 = LSTM(96,  # Best parameter for input_units_lstm2
                      kernel_regularizer=l2(0.052415156927544435),  # Best parameter for l2_reg2
                      recurrent_regularizer=l2(0.052415156927544435),  # Same as kernel regularizer
                      return_sequences=True)(lstm_out)
    
    # Dense layer for feature extraction
    dense_out = Dense(256,  # Best parameter for dense_units
                      activation='relu')(lstm_out_2)
    dropout_out = Dropout(0.2)(dense_out)  # Best parameter for dropout_rate
    
    # Attention mechanism
    attention = Attention()([dropout_out, dropout_out])
    context_vector = tf.reduce_sum(attention, axis=1)
    
    # Output layer
    output = Dense(5, activation='softmax')(context_vector)
    
    # Model definition
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.003964886305018653),  # Best parameter for learning_rate
                  metrics=['accuracy'])
    
    return model
 """

" from keras.models import Model\nfrom keras.layers import Input, LSTM, Dense, Dropout, Attention, Conv1D, MaxPooling1D, BatchNormalization\nfrom keras.regularizers import l2\nimport tensorflow as tf\n\ndef lstm_attention_sensor():\n    inputs = Input(shape=(151, 3))  # Input shape adjusted for accelerometer and gyroscope data\n    \n    # CNN Layer: Extract local features\n    conv_out = Conv1D(filters=96,  # Best parameter for conv_filters\n                      kernel_size=5,  # Best parameter for kernel_size\n                      activation='relu')(inputs)\n    conv_out = MaxPooling1D(pool_size=2)(conv_out)\n    conv_out = BatchNormalization()(conv_out)  # Normalize after convolution\n    \n    # First LSTM layer\n    lstm_out = LSTM(96,  # Best parameter for input_units_lstm1\n                    kernel_regularizer=l2(0.0014023399602325875),  # Best parameter for l2_reg1\n                    recurrent_regularizer=l2(0.0014023399602325875),  # Same as kernel regularizer\n         

In [167]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Attention, Conv1D, MaxPooling1D, BatchNormalization
from keras.regularizers import l2
import tensorflow as tf

def lstm_attention_sensor(hp):
    inputs = Input(shape=(151, 3))  # Input shape adjusted for accelerometer and gyroscope data
    
    # CNN Layer: Extract local features
    conv_out = Conv1D(filters=hp.Int('conv_filters', min_value=16, max_value=256, step=16), 
                      kernel_size=hp.Choice('kernel_size', [3, 5]), 
                      activation='relu')(inputs)
    conv_out = MaxPooling1D(pool_size=2)(conv_out)
    conv_out = BatchNormalization()(conv_out)  # Normalize after convolution
    
    # First LSTM layer
    lstm_out = LSTM(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), 
                    kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    return_sequences=True)(conv_out)
    
    # Second LSTM layer
    lstm_out_2 = LSTM(hp.Int("input_units_lstm2", min_value=32, max_value=256, step=32), 
                      kernel_regularizer=l2(hp.Float('l2_reg2', min_value=0.001, max_value=0.1, sampling='log')),
                      recurrent_regularizer=l2(hp.Float('l2_reg2', min_value=0.001, max_value=0.1, sampling='log')),
                      return_sequences=True)(lstm_out)
    
    # Dense layer for feature extraction
    dense_out = Dense(hp.Int("dense_units", min_value=32, max_value=256, step=32), activation='relu')(lstm_out_2)
    dropout_out = Dropout(hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1))(dense_out)
    
    # Attention mechanism
    attention = Attention()([dropout_out, dropout_out])
    context_vector = tf.reduce_sum(attention, axis=1)
    
    # Output layer
    output = Dense(5, activation='softmax')(context_vector)
    
    # Model definition
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(hp.Float('learning_rate', 
                                                              min_value=1e-4, 
                                                              max_value=1e-2, 
                                                              sampling='LOG', 
                                                              default=1e-3)),
                  metrics=['accuracy'])
    
    return model


In [168]:
#COMENTADO EM 22/11/204
""" from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Attention
from keras.regularizers import l2
import tensorflow as tf

def lstm_attention_sensor(hp):
    inputs = Input(shape=(256, 9))
    
    # LSTM layer
    lstm_out = LSTM(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), 
                    kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    return_sequences=True)(inputs)
    
    
    dense_out = Dense(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), activation='relu')(lstm_out)
    # Attention mechanism using Keras Attention layer
    attention = Attention()([dense_out, dense_out])
    
    # Context vector computation
    context_vector = tf.reduce_sum(attention, axis=1)
    
    # Output layer
    output = Dense(6, activation='softmax')(context_vector)
    
    # Model definition
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(hp.Float('learning_rate', 
                                                              min_value=1e-4, 
                                                              max_value=1e-2, 
                                                              sampling='LOG', 
                                                              default=1e-3)),
                  metrics=['accuracy'])
    
    return model """


' from keras.models import Model\nfrom keras.layers import Input, LSTM, Dense, Dropout, Attention\nfrom keras.regularizers import l2\nimport tensorflow as tf\n\ndef lstm_attention_sensor(hp):\n    inputs = Input(shape=(256, 9))\n    \n    # LSTM layer\n    lstm_out = LSTM(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), \n                    kernel_regularizer=l2(hp.Float(\'l2_reg1\', min_value=0.001, max_value=0.1, sampling=\'log\')),\n                    recurrent_regularizer=l2(hp.Float(\'l2_reg1\', min_value=0.001, max_value=0.1, sampling=\'log\')),\n                    return_sequences=True)(inputs)\n    \n    \n    dense_out = Dense(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), activation=\'relu\')(lstm_out)\n    # Attention mechanism using Keras Attention layer\n    attention = Attention()([dense_out, dense_out])\n    \n    # Context vector computation\n    context_vector = tf.reduce_sum(attention, axis=1)\n    \n    # Output layer\n    out

In [169]:

""" from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Activation, Permute, Multiply, Lambda, Reshape
import keras.backend as K

def lstm_attention_sensor(hp):
    inputs = Input(shape=(256,6))
    lstm_out = LSTM(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
             recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),return_sequences=True)(inputs)
    
    # Attention mechanism
    attention = Dense(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), activation='relu')(lstm_out)    
    attention = Permute([2, 1])(attention)
    attention = Activation('softmax')(attention)
    attention = Permute([2, 1])(attention)
    
    context_vector = Multiply()([lstm_out, attention])
    context_vector = Lambda(lambda x: K.sum(x, axis=1))(context_vector)    
    output = Dense(6, activation='softmax')(context_vector)
    
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
    return model
 """

' from keras.models import Model\nfrom keras.layers import Input, LSTM, Dense, Dropout, Activation, Permute, Multiply, Lambda, Reshape\nimport keras.backend as K\n\ndef lstm_attention_sensor(hp):\n    inputs = Input(shape=(256,6))\n    lstm_out = LSTM(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), kernel_regularizer=l2(hp.Float(\'l2_reg1\', min_value=0.001, max_value=0.1, sampling=\'log\')),\n             recurrent_regularizer=l2(hp.Float(\'l2_reg1\', min_value=0.001, max_value=0.1, sampling=\'log\')),return_sequences=True)(inputs)\n    \n    # Attention mechanism\n    attention = Dense(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), activation=\'relu\')(lstm_out)    \n    attention = Permute([2, 1])(attention)\n    attention = Activation(\'softmax\')(attention)\n    attention = Permute([2, 1])(attention)\n    \n    context_vector = Multiply()([lstm_out, attention])\n    context_vector = Lambda(lambda x: K.sum(x, axis=1))(context_vector)    \n    output

In [170]:
tuner = RandomSearch(
    lstm_attention_sensor,
    objective = 'val_accuracy', 
    seed=42,
    max_trials = 5,
    overwrite=True,        
    executions_per_trial = 1,    
    directory = LOG_DIR,
    project_name="lstm_acel_giro_multiclass")

In [171]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping,reduce_lr,tensorboard]  


In [172]:
from sklearn.utils.class_weight import compute_class_weight

# Assuming y_train is one-hot encoded, convert it to integer labels
y_train_labels = np.argmax(y_train, axis=1)  # Converts one-hot to class indices

# Compute class weights based on the integer labels
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)

# Convert the class weights into a dictionary
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

print(class_weight_dict.values())

dict_values([0.99602872834812, 1.0002545608824778, 1.0022389117188608, 0.9991241700805199, 1.0023809523809524])


In [173]:
""" model = lstm_attention_sensor()
model.fit(x=X_train,
            y=y_train,
            epochs=50,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=64, use_multiprocessing=True) """

' model = lstm_attention_sensor()\nmodel.fit(x=X_train,\n            y=y_train,\n            epochs=50,\n            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=64, use_multiprocessing=True) '

In [174]:
tuner.search(x=X_train,
            y=y_train,
            epochs=50,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=64, use_multiprocessing=True)


Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
176               |176               |conv_filters
3                 |3                 |kernel_size
128               |128               |input_units_lstm1
0.0034974         |0.0034974         |l2_reg1
256               |256               |input_units_lstm2
0.0050547         |0.0050547         |l2_reg2
160               |160               |dense_units
0.1               |0.1               |dropout_rate
0.00098872        |0.00098872        |learning_rate

Epoch 1/50
 38/553 [=>............................] - ETA: 4:57 - loss: 7.5737 - accuracy: 0.4252

In [ ]:
tuner.results_summary()

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.history

In [ ]:
best_model.summary()

In [ ]:
loss, accuracy = best_model.evaluate(X_validation, y_validation)

In [ ]:
predictions = best_model.predict(X_validation)

In [ ]:
print(y_validation)

In [ ]:
print(y_validation.shape)

In [ ]:
y_validation_inverse = np.argmax(y_validation, axis=1)

In [ ]:
print(y_validation_inverse)

In [ ]:
print(len(y_validation_inverse))

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
print(classes_x.shape)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report,accuracy_score
cm = confusion_matrix(y_validation_inverse, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report,accuracy_score
# Calculate accuracy for each class
class_accuracies = {}
for cls in np.unique(classes_x):
    class_mask = (y_validation_inverse == cls)
    class_accuracies[f"Class {cls}"] = accuracy_score(y_validation_inverse[class_mask], classes_x[class_mask])

# Print class-wise accuracies
print("Accuracy for each class:")
for cls, acc in class_accuracies.items():
    print(f"{cls}: {acc:.2f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Example function to calculate per-class accuracy
def calculate_per_class_accuracy(y_true, y_pred, class_names=None):
    """
    Calculate accuracy for each class.
    
    Parameters:
    - y_true: Ground truth labels (list or numpy array).
    - y_pred: Predicted labels (list or numpy array).
    - class_names: List of class names (optional).
    
    Returns:
    - A dictionary with class-wise accuracies.
    """
    # Generate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate accuracy for each class
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
    
    # Organize results
    results = {}
    if class_names is None:
        class_names = [f'Class {i}' for i in range(len(per_class_accuracy))]
    for class_name, accuracy in zip(class_names, per_class_accuracy):
        results[class_name] = accuracy
    
    return results

# Example usage
if __name__ == "__main__":
    # Example true and predicted labels
    #y_true = [0, 1, 2, 2, 0, 1, 1, 0, 2, 1]  # Ground truth
    #y_pred = [0, 1, 1, 2, 0, 1, 2, 0, 2, 1]  # Predictions
    
    # Class names (optional)
    class_names = [0, 1, 2,3,4,5,6]
    
    # Calculate and display per-class accuracy
    accuracy_per_class = calculate_per_class_accuracy(y_validation_inverse, classes_x, class_names)
    for class_name, accuracy in accuracy_per_class.items():
        print(f"{class_name}: {accuracy:.2f}")
    
    # Display class


In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report,accuracy_score
# Calculate accuracy for each class
class_accuracies = {}
for cls in np.unique(classes_x):
    class_mask = (y_validation_inverse == cls)
    class_accuracies[f"Class {cls}"] = accuracy_score(y_validation_inverse[class_mask], classes_x[class_mask])

# Print class-wise accuracies
print("Accuracy for each class:")
for cls, acc in class_accuracies.items():
    print(f"{cls}: {acc:.2f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Example function to calculate per-class accuracy
def calculate_per_class_accuracy(y_true, y_pred, class_names=None):
    """
    Calculate accuracy for each class.
    
    Parameters:
    - y_true: Ground truth labels (list or numpy array).
    - y_pred: Predicted labels (list or numpy array).
    - class_names: List of class names (optional).
    
    Returns:
    - A dictionary with class-wise accuracies.
    """
    # Generate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate accuracy for each class
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
    
    # Organize results
    results = {}
    if class_names is None:
        class_names = [f'Class {i}' for i in range(len(per_class_accuracy))]
    for class_name, accuracy in zip(class_names, per_class_accuracy):
        results[class_name] = accuracy
    
    return results

# Example usage
if __name__ == "__main__":
    # Example true and predicted labels
    #y_true = [0, 1, 2, 2, 0, 1, 1, 0, 2, 1]  # Ground truth
    #y_pred = [0, 1, 1, 2, 0, 1, 2, 0, 2, 1]  # Predictions
    
    # Class names (optional)
    class_names = [0, 1, 2,3,4,5,6]
    
    # Calculate and display per-class accuracy
    accuracy_per_class = calculate_per_class_accuracy(y_validation_inverse, classes_x, class_names)
    for class_name, accuracy in accuracy_per_class.items():
        print(f"{class_name}: {accuracy:.2f}")
    
    # Display class


In [ ]:
best_model.save(WORKDIR+"/Model/best_models_saved_"+data_hora)